# Fase 5 — Datathon Passos Mágicos: limpeza e preparação do painel PEDE

Este notebook é o primeiro passo da Fase 5 (Datathon *"Case Passos Mágicos"*). O objetivo aqui é só **preparar os dados**: partir do arquivo bruto `data/raw/base_bronze.xlsx`, 3 abas (`PEDE2022`, `PEDE2023`, `PEDE2024`), uma por edição da Pesquisa Extensiva do Desenvolvimento Educacional (PEDE) aplicada aos alunos da Associação Passos Mágicos, identificados de forma anônima e estável entre os anos pela coluna `RA` — e chegar em um painel longo, harmonizado e validado, salvo em `data/processed/pede_painel_consolidado.csv`.

Toda a lógica de harmonização vive em `src/data_prep.py`; este notebook só importa e usa essas funções.


In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src.data_prep import (
    RANDOM_STATE,
    build_painel,
    build_painel_com_alvo,
    calcular_pedra_por_inde,
    carregar_abas_brutas,
    inspecionar_abas,
    padronizar_nomes_e_categorias,
)

pd.set_option("display.max_columns", 60)
RANDOM_STATE

42

## 1. Inspeção das 3 abas 

Primeira observação da base bruta, por aba.

In [2]:
abas_brutas = carregar_abas_brutas("../data/raw/base_bronze.xlsx")
resumo_abas = inspecionar_abas(abas_brutas, verbose=True)

PEDE2022: 860 linhas x 42 colunas
['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA', 'Matem', 'Portug', 'Inglês', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV']
PEDE2023: 1014 linhas x 48 colunas
['RA', 'Fase', 'INDE 2023', 'Pedra 2023', 'Turma', 'Nome Anonimizado', 'Data de Nasc', 'Idade', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'INDE 22', 'INDE 23', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'IPP', 'Rec Psicologia', 'IDA', 'Mat', 'Por', 'Ing', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase 

**O que a inspeção acima mostra:**

- As 3 abas têm números de colunas diferentes (42, 48 e 50) e nomes que mudam
  de ano para ano, mesmo quando representam o mesmo conceito. Por exemplo, o
  INDE do próprio ano é a coluna `INDE 22` em `PEDE2022`, mas `INDE 2023` em
  `PEDE2023` e `INDE 2024` em `PEDE2024` (as colunas de 2 dígitos `INDE 23`
  e `Pedra 23` que aparecem em `PEDE2023` são um campo vazio, não o
  valor do ano).
- `IPP` só aparece nas colunas de `PEDE2023` e `PEDE2024`, não existindo em
  `PEDE2022`. Esse indicador passou a ser medido a partir de 2023.
- A coluna de Fase muda de formato a cada ano: inteiro puro em `PEDE2022`
  (`Fase` = 0-7), texto `"ALFA"`/`"FASE N"` em `PEDE2023`, e em `PEDE2024`
  passa a embutir a turma no valor (`"1A"`, `"2B"`, ...) ou vir só como
  número solto (`"9"`). A função `_extrair_numero_fase` em `src/data_prep.py`
  documenta e trata cada um desses formatos.
- `Fase ideal` (2022) e `Fase Ideal` (2023/2024) também têm capitalização
  diferente do nome da coluna.


## 2. Harmonização: `build_painel()`

`build_painel()` (em `src/data_prep.py`) faz, para cada ano:

1. Renomeia as colunas relevantes de cada aba para um esquema comum em
   snake_case (`ra`, `fase_num`, `inde`, `pedra`, `ian`, `ida`, `ieg`, `iaa`,
   `ips`, `ipp`, `ipv`, notas, dados demográficos etc.), resolvendo
   explicitamente o nome de origem por ano (não por um padrão único, porque
   não existe um padrão único).
2. Extrai o nível numérico de `fase_num` e `fase_ideal_num` a partir do
   formato específico de cada ano (ver `_extrair_numero_fase`).
3. Calcula `defasagem_calculada = fase_num - fase_ideal_num`.
4. Empilha os 3 anos em uma base longa (1 linha por aluno-ano) com a
   coluna `ano` identificando 2022/2023/2024.


In [3]:
painel = build_painel("../data/raw/base_bronze.xlsx", verbose=False)

n_linhas_brutas = sum(len(df) for df in abas_brutas.values())
print(f"Linhas nas 3 abas brutas somadas: {n_linhas_brutas}")
print(f"Linhas no painel harmonizado:      {len(painel)}")
print(f"Shape do painel: {painel.shape}")
print()
print("Alunos (RA) distintos:", painel["ra"].nunique())
print("Anos presentes:", sorted(painel["ano"].unique().tolist()))
print()
painel.head()

Linhas nas 3 abas brutas somadas: 3030
Linhas no painel harmonizado:      3030
Shape do painel: (3030, 26)

Alunos (RA) distintos: 1661
Anos presentes: [2022, 2023, 2024]



,ra,ano,fase_num,fase_ideal_num,defasagem_fornecida,inde,pedra,ian,ida,ieg,iaa,ips,ipp,ipv,nota_matematica,nota_portugues,nota_ingles,indicado,atingiu_pv,turma,genero,ano_ingresso,instituicao_ensino,ano_nascimento,defasagem_calculada,idade_anos
0,1,2022,7.0,8.0,-1,5.783,Quartzo,5.0,4.0,4.1,8.3,5.6,NaN,7.278,2.7,3.5,6.0,Sim,Não,A,Feminino,2016,Escola Pública,2003,-1.0,19
1,1,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,Feminino,2016,Privada *Parcerias com Bolsa 100%,2003,0.0,20
2,1,2024,8.0,8.0,0,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,Feminino,2021,Privada *Parcerias com Bolsa 100%,2003,0.0,21
3,2,2022,7.0,7.0,0,7.055,Ametista,10.0,6.8,5.2,8.8,6.3,NaN,6.778,6.3,4.5,9.7,Não,Não,A,Feminino,2017,Rede Decisão,2005,0.0,17
4,2,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,Feminino,2017,Privada *Parcerias com Bolsa 100%,2005,0.0,18


In [4]:
print("Tipos de dado por coluna:")
painel.dtypes

Tipos de dado por coluna:


ra                       int64
ano                      int64
fase_num               float64
fase_ideal_num         float64
defasagem_fornecida      int64
inde                   float64
pedra                      str
ian                    float64
ida                    float64
ieg                    float64
iaa                    float64
ips                    float64
ipp                    float64
ipv                    float64
nota_matematica        float64
nota_portugues         float64
nota_ingles            float64
indicado                object
atingiu_pv              object
turma                   object
genero                     str
ano_ingresso             int64
instituicao_ensino         str
ano_nascimento           int64
defasagem_calculada    float64
idade_anos               int64
dtype: object

## 3. QA — `defasagem_calculada` vs. a defasagem já fornecida pela Passos Mágicos

A Passos Mágicos já fornece uma coluna de defasagem pronta (`Defas` em 2022,
`Defasagem` em 2023/2024, harmonizada aqui como `defasagem_fornecida`).
Comparamos com `defasagem_calculada` (a partir de `fase_num - fase_ideal_num`)
para validar se a extração de fase está correta.

In [5]:
comparavel = painel.dropna(subset=["defasagem_calculada", "defasagem_fornecida"])
taxa_match = (comparavel["defasagem_calculada"] == comparavel["defasagem_fornecida"]).mean()
correlacao = comparavel["defasagem_calculada"].corr(comparavel["defasagem_fornecida"])

print(f"Linhas comparáveis (ambas as colunas preenchidas): {len(comparavel)} de {len(painel)}")
print(f"% de linhas com defasagem_calculada == defasagem_fornecida: {taxa_match:.2%}")
print(f"Correlação (Pearson) entre as duas: {correlacao:.4f}")

divergentes = comparavel[comparavel["defasagem_calculada"] != comparavel["defasagem_fornecida"]]
print(f"\nLinhas divergentes: {len(divergentes)}")
divergentes[["ra", "ano", "fase_num", "fase_ideal_num", "defasagem_calculada", "defasagem_fornecida"]]

Linhas comparáveis (ambas as colunas preenchidas): 3030 de 3030
% de linhas com defasagem_calculada == defasagem_fornecida: 99.93%
Correlação (Pearson) entre as duas: 0.9960

Linhas divergentes: 2


,ra,ano,fase_num,fase_ideal_num,defasagem_calculada,defasagem_fornecida
2884,1516,2024,3.0,3.0,0.0,3
2887,1519,2024,3.0,3.0,0.0,3


**Leitura do resultado acima:** a concordância é muito alta, e isso valida que `_extrair_numero_fase`
está lendo corretamente os três formatos de coluna de Fase. As poucas linhas
divergentes (uma fração mínima da base) têm `fase_num == fase_ideal_num`
(ou seja, `defasagem_calculada = 0`) mas uma `defasagem_fornecida` diferente
de zero. O padrão é consistente com uma inconsistência pontual de
digitação na fonte, não com um erro sistemático de parsing (se fosse erro de
parsing, esperaríamos divergências concentradas em um ano/formato
específico, e não é o que vemos).

## 4. QA — Pedra recalculada a partir do INDE vs. Pedra fornecida

Recalculamos a Pedra de cada aluno a partir do `inde` harmonizado, usando as
faixas oficiais (`calcular_pedra_por_inde`, que usa `FAIXAS_PEDRA_POR_INDE`:
Quartzo 2,405–5,506 | Ágata 5,506–6,868 | Ametista 6,868–8,230 | Topázio
8,230–9,294), e comparamos com a Pedra que a Passos Mágicos já forneceu.

In [6]:
comparavel_pedra = painel.dropna(subset=["inde", "pedra"]).copy()
comparavel_pedra["pedra_recalculada"] = calcular_pedra_por_inde(comparavel_pedra["inde"])

taxa_match_pedra = (comparavel_pedra["pedra"] == comparavel_pedra["pedra_recalculada"]).mean()
print(f"Linhas comparáveis (INDE e Pedra preenchidos): {len(comparavel_pedra)} de {len(painel)}")
print(f"% de linhas com Pedra recalculada == Pedra fornecida: {taxa_match_pedra:.2%}")
print()
print("Crosstab Pedra fornecida (linha) x Pedra recalculada (coluna):")
pd.crosstab(comparavel_pedra["pedra"], comparavel_pedra["pedra_recalculada"])

Linhas comparáveis (INDE e Pedra preenchidos): 2845 de 3030
% de linhas com Pedra recalculada == Pedra fornecida: 81.51%

Crosstab Pedra fornecida (linha) x Pedra recalculada (coluna):


pedra_recalculada,Ametista,Quartzo,Topázio,Ágata
pedra,,,,
Ametista,1120,0,0,0
Quartzo,0,149,0,167
Topázio,219,0,457,0
Ágata,128,0,0,593


**Leitura do resultado acima:** o match é bem mais baixo que na seção 3 (na
faixa de 80%, não de 99%), mas o crosstab mostra que isso **não** é ruído
aleatório, é um padrão sistemático de fronteira: `Ametista` bate 100% das
vezes, enquanto uma fatia relevante dos alunos com Pedra fornecida
`Quartzo` recalcula como `Ágata`, e uma fatia relevante dos `Topázio`
recalcula como `Ametista`. Ou seja, as divergências se concentram exatamente
nas bordas entre faixas vizinhas. A leitura mais provável é que a Passos
Mágicos aplica, na prática, um critério de corte com alguma variação por
ciclo/ano (arredondamento, ajuste de faixa, ou uma versão ligeiramente
diferente das faixas documentadas) que o documento consultado não captura
integralmente, e as faixas oficiais continuam sendo uma boa aproximação
(a maioria das linhas bate, e a Pedra cresce monotonicamente com o INDE),
mas não devem ser tratadas como uma fórmula exata e imutável.

## 5. Mapeamento de dados ausentes por coluna e por ano

Olhamos a % de ausência de cada coluna, separada por ano, para diferenciar
"ausência estrutural" (indicador que só passou a existir/ser preenchido a
partir de determinado ano) de ausência real de resposta.

In [7]:
percentual_ausente_por_ano = pd.concat(
    {
        ano: painel.loc[painel["ano"] == ano].drop(columns=["ra", "ano"]).isna().mean()
        for ano in sorted(painel["ano"].unique())
    },
    axis=1,
).round(3)
percentual_ausente_por_ano

,2022,2023,2024
fase_num,0.000,0.000,0.000
fase_ideal_num,0.000,0.000,0.000
defasagem_fornecida,0.000,0.000,0.000
inde,0.000,0.082,0.088
pedra,0.000,0.082,0.088
ian,0.000,0.000,0.000
ida,0.000,0.076,0.087
ieg,0.000,0.075,0.000
iaa,0.000,0.062,0.088
ips,0.000,0.068,0.088


**Leitura do mapa de ausência acima:**

- `ipp`: **100% ausente em 2022** e majoritariamente preenchido em 2023/2024 confirma o que o indicador só passou
  a ser medido a partir de 2023, não é um problema de harmonização.
- `indicado` e `atingiu_pv`: o oposto, preenchidos em 2022 e **100% ausentes em 2023 e 2024** (as colunas existem nessas abas, mas vêm totalmente vazias). Esses dois indicadores só foram registrados na edição de 2022 da pesquisa.
- `nota_ingles` tem uma taxa de ausência alta (bem acima de 50%) **nos três anos** — diferente dos casos acima, essa ausência não é específica de um ano, então não é um indicador que "passou a existir depois": é mais provável que a avaliação de inglês nem sempre seja aplicada a todos os alunos (ex.: fases iniciais sem ensino de inglês), e isso deve ser tratado como ausência real na fonte, não como falha de harmonização.
- Os indicadores centrais (`inde`, `ida`, `ieg`, `iaa`, `ips`, `ipp`, `ipv`, notas) têm ausência baixa mas não nula em 2023/2024 (uma fração pequena de avaliações incompletas por aluno) e praticamente nula em 2022.

In [8]:
componentes_inde = ["ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv"]

inde_ausente = painel[painel["inde"].isna()]
com_ipp_presente = inde_ausente[inde_ausente["ipp"].notna()]

print(f"Linhas com INDE ausente: {len(inde_ausente)}")
print(f"Dessas, com IPP presente: {len(com_ipp_presente)}")
print()
print("Ausência (True) dos demais componentes do INDE nas linhas com INDE ausente e IPP presente:")
com_ipp_presente[componentes_inde].isna()

Linhas com INDE ausente: 185
Dessas, com IPP presente: 7

Ausência (True) dos demais componentes do INDE nas linhas com INDE ausente e IPP presente:


,ian,ida,ieg,iaa,ips,ipp,ipv
300,False,False,False,False,True,False,False
534,False,False,False,False,True,False,False
551,False,False,False,False,True,False,False
554,False,False,False,False,True,False,False
2390,False,True,False,False,False,False,False
2492,False,False,False,False,True,False,False
2494,False,False,False,False,True,False,False


**Achado — `ipp` não ajuda a recalcular o INDE faltante:**

- `ipp` está **100% ausente em 2022** (indicador não existia nesse ano) e também ausente em uma fração de 2023/2024. Essa ausência em 2023/2024 não é específica das linhas com INDE faltante.
- Das 185 linhas com `inde` ausente, só 7 têm `ipp` preenchido e nessas 7, `ipp` nunca é o único componente faltando: o que falta é sempre `ida` ou `ips`. Ou seja, mesmo nas poucas linhas em que o IPP está disponível, não dá pra recompor o INDE a partir dos 7 componentes porque sempre falta outro.
- Chegamos a cogitar isolar o IPP faltante de 2022 algebricamente a partir do INDE + os outros 6 indicadores (IAN, IDA, IEG, IAA, IPS, IPV), já que o INDE é uma combinação conhecida desses 7 componentes. Descartamos a ideia por dois motivos: (1) essa fórmula de 7 componentes só foi validada contra dados reais de 2023/2024; e (2) mesmo que valesse, o IPP "recuperado" seria só uma combinação linear das colunas que já temos no painel, sem trazer nenhuma informação nova.
- A decisão de como tratar `ipp` no modelo preditivo (dropar a coluna, imputar, ou usar um indicador binário de disponibilidade) fica para o notebook de modelagem.

## 6. Alvo: `build_painel_com_alvo()`

`build_painel_com_alvo()` cria `alvo_risco_defasagem_prox_ano`: 1 se o aluno
está defasado (`defasagem_calculada < 0`) no ano **seguinte** ao da linha, 0
se não está, e `NA` se o aluno não tem registro no ano seguinte. O alvo vem
deliberadamente do ano seguinte ao das features de cada linha (join em
`ra` + `ano - 1`), para não vazar informação do próprio ano.

Explicação mais detalhada:

O objetivo do modelo não é descrever a situação atual do aluno, pois isso a gente já sabe (defasagem_calculada). O objetivo é prever o futuro. Dado o que se sabe do aluno hoje, ele vai estar defasado no ano que vem? 
defasagem_calculada é simples: fase_num - fase_ideal_num. Se for negativo, o aluno está atrás do esperado (defasado); se for zero ou positivo, está em dia ou à frente.

Por que não posso simplesmente usar a defasagem do mesmo ano como alvo
Se eu treinasse um modelo pra prever "esse aluno está defasado" usando dados do mesmo ano como pergunta e resposta, não seria previsão nenhuma, seria só descrição do presente, e o modelo teria acesso direto a informação que já entrega a resposta (vazamento de dados). O valor de um modelo preditivo está em usar informação de hoje pra estimar algo que só vai acontecer depois (defasagem no ano N+1). Assim ele serve pra agir preventivamente, antes do problema acontecer.

O exemplo real (RA 1):
Ano	    defasagem_calculada	    alvo_risco_defasagem_prox_ano
2022	 -1 (defasado)	         0
2023	 0 (em dia)	             0
2024	 0 (em dia)	             NaN

A linha de 2022 tem defasagem -1 (o aluno estava defasado naquele ano), mas o alvo dela é 0, não 1. Por quê? Porque o alvo da linha de 2022 não olha pra defasagem de 2022, ele olha pra defasagem de 2023 (o ano seguinte), que foi 0 (em dia). Ou seja: "esse aluno, que em 2022 estava defasado, deixou de estar defasado em 2023" → alvo = 0 (não entrou em risco no ano seguinte). Já a linha de 2024 fica com alvo NaN porque não existe PEDE2025 na base pra conferir o que aconteceu depois — não tem como saber.

In [9]:
painel_com_alvo = build_painel_com_alvo(painel)

print(f"Shape do painel com alvo: {painel_com_alvo.shape}")
print(f"(painel original tinha {painel.shape[0]} linhas — build_painel_com_alvo não deve alterar o nº de linhas)\n")

for ano in sorted(painel_com_alvo["ano"].unique()):
    fatia_ano = painel_com_alvo.loc[painel_com_alvo["ano"] == ano, "alvo_risco_defasagem_prox_ano"]
    n_total = len(fatia_ano)
    n_definido = fatia_ano.notna().sum()
    print(f"ano={ano}: {n_definido}/{n_total} alunos ({n_definido / n_total:.1%}) com alvo definido")
    if n_definido > 0:
        distrib = fatia_ano.value_counts(normalize=True).sort_index()
        print(f"    dentre os definidos -> {distrib.to_dict()}")

Shape do painel com alvo: (3030, 27)
(painel original tinha 3030 linhas — build_painel_com_alvo não deve alterar o nº de linhas)

ano=2022: 600/860 alunos (69.8%) com alvo definido
    dentre os definidos -> {0.0: 0.39, 1.0: 0.61}
ano=2023: 765/1014 alunos (75.4%) com alvo definido
    dentre os definidos -> {0.0: 0.5973856209150327, 1.0: 0.40261437908496733}
ano=2024: 0/1156 alunos (0.0%) com alvo definido


**Leitura da distribuição acima:** para `ano == 2024` não existe alvo definido para nenhum aluno, e é o comportamento esperado, porque não há `PEDE2025` nesta base para calcular a defasagem do "ano seguinte" (não é um bug). Para 2022 e 2023, nem todos os alunos têm alvo definido, porque nem todo `RA` de um ano reaparece no ano seguinte (evasão, novos ingressos etc.), mas a maioria tem, o que deixa uma base de treino razoável para as duas safras com alvo.

## 7. Padronização final: nomes de coluna e categorias

Último passo antes de salvar o CSV: `padronizar_nomes_e_categorias()` (em
`src/data_prep.py`) recebe `painel_com_alvo` e:

1. Renomeia todas as colunas para maiúsculo.
2. Remove caracteres especiais.


In [11]:
painel_final = padronizar_nomes_e_categorias(painel_com_alvo)

painel_final.head(5)

,RA,ANO,FASE,FASE_IDEAL,DEFASAGEM_FORNECIDA,INDE,PEDRA,IAN,IDA,IEG,IAA,IPS,IPP,IPV,NOTA_MATEMATICA,NOTA_PORTUGUES,NOTA_INGLES,INDICADO,ATINGIU_PV,TURMA,GENERO,ANO_INGRESSO,INSTITUICAO_ENSINO,ANO_NASCIMENTO,DEFASAGEM_CALCULADA,IDADE_ANOS,ALVO_RISCO_DEFASAGEM_PROX_ANO
0,1,2022,7.0,8.0,-1,5.783,QUARTZO,5.0,4.0,4.1,8.3,5.6,NaN,7.278,2.7,3.5,6.0,SIM,NAO,A,FEMININO,2016,ESCOLA PUBLICA,2003,-1.0,19,0.0
1,1,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,FEMININO,2016,PRIVADA PARCERIAS COM BOLSA 100,2003,0.0,20,0.0
2,1,2024,8.0,8.0,0,NaN,NaN,10.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,FEMININO,2021,PRIVADA PARCERIAS COM BOLSA 100,2003,0.0,21,NaN
3,2,2022,7.0,7.0,0,7.055,AMETISTA,10.0,6.8,5.2,8.8,6.3,NaN,6.778,6.3,4.5,9.7,NAO,NAO,A,FEMININO,2017,REDE DECISAO,2005,0.0,17,0.0
4,2,2023,8.0,8.0,0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8E,FEMININO,2017,PRIVADA PARCERIAS COM BOLSA 100,2005,0.0,18,0.0


**Decisões deliberadas sobre o CSV final:**

- **Nenhuma coluna numérica é arredondada.** `painel_final` mantém todas as casas decimais como o pandas já representa (float nativo) — não cortamos precisão em `INDE`, `IDA`, notas etc.
- **O separador decimal continua ponto**, o padrão do `to_csv` do pandas, nunca vírgula. Os modelos de ML da fase de modelagem (scikit-learn) esperam ponto nativamente.
- **Nenhum valor ausente é preenchido** com placeholder (nem `"NULO"`, nem `0`, nem string vazia tratada como categoria). Célula ausente continua ausente (`NaN`), porque preencher com qualquer valor faria os modelos e as contagens de ausência (já documentadas na seção 5, QA) tratarem "ausência" como se fosse um dado real, o que distorceria tanto a EDA quanto o modelo.

## 8. Salvando o painel consolidado

In [ ]:
from pathlib import Path

caminho_saida = Path("../data/processed/pede_painel_consolidado.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)
painel_final.to_csv(caminho_saida, index=False)

print(f"Salvo em: {caminho_saida.resolve()}")
print(f"Existe no disco? {caminho_saida.exists()}")
print(f"Tamanho do arquivo: {caminho_saida.stat().st_size:,} bytes")

conferencia = pd.read_csv(caminho_saida)
print(f"\nRelido do disco: shape={conferencia.shape} (esperado: {painel_final.shape})")
assert conferencia.shape == painel_final.shape, "Shape do CSV relido não bate com o painel em memória"
assert list(conferencia.columns) == list(painel_final.columns), "Colunas do CSV relido não batem com painel_final"

Salvo em: C:\Users\maria\OneDrive\Documentos\tech-challenge-passos-magicos\data\processed\pede_painel_consolidado.csv
Existe no disco? True
Tamanho do arquivo: 410,910 bytes

Relido do disco: shape=(3030, 27) (esperado: (3030, 27))


## 9. Resumo

In [ ]:
n_ipp_ausente_2022 = painel.loc[painel["ano"] == 2022, "ipp"].isna().mean()
n_ipp_preenchido_2023 = painel.loc[painel["ano"] == 2023, "ipp"].notna().mean()
prop_alvo_2022 = painel_com_alvo.loc[painel_com_alvo["ano"] == 2022, "alvo_risco_defasagem_prox_ano"].notna().mean()
prop_alvo_2023 = painel_com_alvo.loc[painel_com_alvo["ano"] == 2023, "alvo_risco_defasagem_prox_ano"].notna().mean()

linhas_resumo = [
    f"- Painel consolidado: {painel_com_alvo.shape[0]} linhas (aluno-ano) x {painel_com_alvo.shape[1]} colunas, "
    f"cobrindo {painel_com_alvo['ra'].nunique()} alunos (RA) distintos nos anos "
    f"{sorted(painel_com_alvo['ano'].unique().tolist())}.",
    f"- QA defasagem: {taxa_match:.2%} de concordância entre defasagem_calculada e defasagem_fornecida "
    f"(correlação {correlacao:.4f}) em {len(comparavel)} linhas comparáveis -> extração de fase validada.",
    f"- QA Pedra: {taxa_match_pedra:.2%} de concordância entre a Pedra recalculada a partir do INDE e a Pedra "
    f"fornecida, em {len(comparavel_pedra)} linhas comparáveis -> divergência concentrada em fronteiras de "
    f"faixa, não em erro de parsing (ver crosstab da seção 4).",
    f"- ipp: {n_ipp_ausente_2022:.0%} ausente em 2022 (indicador não existia) vs. "
    f"{n_ipp_preenchido_2023:.0%} preenchido em 2023.",
    f"- Alvo (alvo_risco_defasagem_prox_ano): definido para {prop_alvo_2022:.1%} dos alunos de 2022 e "
    f"{prop_alvo_2023:.1%} dos de 2023; indefinido (NA) para 100% de 2024, por não existir PEDE2025 na base.",
    f"- Arquivo salvo em data/processed/pede_painel_consolidado.csv ({caminho_saida.stat().st_size:,} bytes), "
    f"relido do disco e conferido contra o DataFrame em memória.",
]
print("\n".join(linhas_resumo))

- Painel consolidado: 3030 linhas (aluno-ano) x 27 colunas, cobrindo 1661 alunos (RA) distintos nos anos [2022, 2023, 2024].
- QA defasagem: 99.93% de concordância entre defasagem_calculada e defasagem_fornecida (correlação 0.9960) em 3030 linhas comparáveis -> extração de fase validada.
- QA Pedra: 81.51% de concordância entre a Pedra recalculada a partir do INDE e a Pedra fornecida, em 2845 linhas comparáveis -> divergência concentrada em fronteiras de faixa, não em erro de parsing (ver crosstab da seção 4).
- ipp: 100% ausente em 2022 (indicador não existia) vs. 93% preenchido em 2023.
- Alvo (alvo_risco_defasagem_prox_ano): definido para 69.8% dos alunos de 2022 e 75.4% dos de 2023; indefinido (NA) para 100% de 2024, por não existir PEDE2025 na base.
- Arquivo salvo em data/processed/pede_painel_consolidado.csv (410,910 bytes), relido do disco e conferido contra o DataFrame em memória.
